# Seminar Project: Bivariate Decision Trees (BiCART & BiTAO)

**Base Research Paper:** *Bivariate Decision Trees: Smaller, Interpretable, More Accurate*  
**Authors:** Rasul Kairgeldin and Miguel Á. Carreira-Perpiñán  
**Conference:** ACM SIGKDD 2024  
**Course:** B.Tech Computer Science & Engineering — Seminar / Research Paper Implementation

---
## Executive Summary
This notebook demonstrates the implementation of **BiCART** and **BiTAO**, two novel bivariate decision tree algorithms proposed at KDD 2024.
- Standard decision trees (CART) split on single features ($x_i \le \theta$), creating axis-parallel hyperplanes.
- Oblique decision trees split on all features ($\mathbf{w}^T \mathbf{x} + b \le 0$), creating dense, uninterpretable models.
- **Bivariate Decision Trees** split on **at most 2 features** ($w_1 x_i + w_2 x_j + b \le 0$), offering higher accuracy with compact, shallow, 2D-interpretable trees.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from preprocessing import get_data
from cart import CARTClassifier
from bicart import BiCARTClassifier
from bitao import BiTAOClassifier
from metrics import compute_all_metrics, print_metrics, print_tree_stats
from visualization import (
    plot_accuracy_comparison, plot_tree_size_comparison,
    plot_confusion_matrix, plot_bivariate_split_illustration
)
print('All modules loaded successfully!')

## Phase 1: Load and Preprocess Dataset
We use the **UCI Breast Cancer Wisconsin (Diagnostic)** dataset (569 instances, 30 continuous features, 2 classes).

In [ ]:
X_train, X_test, y_train, y_test, feature_names, class_names = get_data(test_size=0.2, random_state=42, scale=True)
print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')
print(f'Feature count: {X_train.shape[1]}')

## Phase 2: Standard Univariate CART
Train standard CART using Gini impurity and univariate splits ($x_i \le \theta$).

In [ ]:
cart_model = CARTClassifier(max_depth=7, min_samples_split=2)
cart_model.fit(X_train, y_train)
cart_metrics = compute_all_metrics(cart_model, X_train, y_train, X_test, y_test)
print_metrics('Standard CART', cart_metrics)
print_tree_stats(cart_metrics)

## Phase 3: BiCART (Bivariate CART)
Train BiCART using greedy top-down search over feature pairs $\binom{D}{2}$ and $H=36$ orientation angles.

In [ ]:
bicart_model = BiCARTClassifier(max_depth=5, num_orientations=36, min_samples_split=2)
bicart_model.fit(X_train, y_train)
bicart_metrics = compute_all_metrics(bicart_model, X_train, y_train, X_test, y_test)
print_metrics('BiCART', bicart_metrics)
print_tree_stats(bicart_metrics)

## Phase 4: BiTAO (Bivariate Tree Alternating Optimization)
Train BiTAO with global alternating optimization and $L_1$-regularization.

In [ ]:
bitao_model = BiTAOClassifier(init_depth=4, lam=1.0, C=1.5, num_orientations=36, max_iter=15, random_state=42)
bitao_model.fit(X_train, y_train)
bitao_metrics = compute_all_metrics(bitao_model, X_train, y_train, X_test, y_test)
print_metrics('BiTAO', bitao_metrics)
print_tree_stats(bitao_metrics)

## Phase 5: Visualizing Results and 2D Interpretability

In [ ]:
metrics_dict = {
    'CART': cart_metrics,
    'BiCART': bicart_metrics,
    'BiTAO': bitao_metrics
}
plot_accuracy_comparison(metrics_dict)
plot_tree_size_comparison(metrics_dict)